## 2. Artifact-Level Analysis

The AV dataset contains repeated detections of the same underlying files.

To avoid treating repeated detections as independent malicious artifacts, we extract:

- endpoint_id
- file_name
- SHA256
- quarantine timestamp

We then deduplicate and examine artifact reuse across endpoints.

In [4]:
import pandas as pd
from pathlib import Path

# Project root = parent of the notebook directory
BASE = Path.cwd().parent
DATA = BASE / "data"

# Load datasets
web = pd.read_csv(DATA / "web_activity.csv")
security = pd.read_csv(DATA / "endpoint_security.csv")
usb = pd.read_csv(DATA / "usb_events.csv")

# Convert timestamps
web["event_time"] = pd.to_datetime(web["event_time"], errors="coerce")
web["received_time"] = pd.to_datetime(web["received_time"], errors="coerce")

security["event_time"] = pd.to_datetime(security["event_time"], errors="coerce")
security["received_time"] = pd.to_datetime(security["received_time"], errors="coerce")

usb["event_time"] = pd.to_datetime(usb["event_time"], errors="coerce")
usb["received_time"] = pd.to_datetime(usb["received_time"], errors="coerce")

print("Web activity:", web.shape)
print("Endpoint security:", security.shape)
print("USB events:", usb.shape)

print("\nSecurity log types:")
print(security["log_type"].value_counts())

Web activity: (2812118, 8)
Endpoint security: (23540, 8)
USB events: (8, 8)

Security log types:
log_type
quarantine    20004
av_scan        2622
web_block       914
Name: count, dtype: int64


C:\Users\Asus\AppData\Local\Temp\ipykernel_5316\2796173385.py:20: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  usb["event_time"] = pd.to_datetime(usb["event_time"], errors="coerce")
C:\Users\Asus\AppData\Local\Temp\ipykernel_5316\2796173385.py:21: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  usb["received_time"] = pd.to_datetime(usb["received_time"], errors="coerce")


In [5]:
quarantine = security[
    security["log_type"] == "quarantine"
].copy()

print("Quarantine records:", len(quarantine))
print("Quarantine endpoints:", quarantine["endpoint_id"].nunique())

Quarantine records: 20004
Quarantine endpoints: 244


In [6]:
malicious = quarantine[
    quarantine["message"].str.contains(
        r"FileType:\s*Malicious",
        case=False,
        na=False
    )
].copy()

print("Raw malicious quarantine records:", len(malicious))
print("Endpoints:", malicious["endpoint_id"].nunique())

Raw malicious quarantine records: 134
Endpoints: 33


In [7]:
# Show all malicious quarantine records
malicious_view = malicious[
    ["endpoint_id", "event_time", "message"]
].sort_values(["endpoint_id", "event_time"])

display(malicious_view)

,endpoint_id,event_time,message
22765,EP2SJ6NE,NaT,File: libreoffice-calc.desktop | Quarantined A...
17283,EP3GALVS,NaT,File: recently-used.xbel | Quarantined At: 202...
17970,EP3GALVS,NaT,File: recently-used.xbel | Quarantined At: 202...
9302,EP4YIFHV,2026-11-09 09:23:33,File: wallet_donation_driver.js | Quarantined ...
9352,EP4YIFHV,2026-11-09 09:23:33,File: load-ec-deps.bundle.js | Quarantined At:...
...,...,...,...
21524,EPVSWBNA,NaT,File: Strings.dll | Quarantined At: 2026-09-23...
21525,EPVSWBNA,NaT,File: launcher.exe | Quarantined At: 2026-09-2...
21526,EPVSWBNA,NaT,File: zshp1020s.dll | Quarantined At: 2026-09-...
22259,EPWZ7L6L,NaT,File: ops-electron-app.desktop | Quarantined A...


In [8]:
malicious_analysis = malicious.copy()

malicious_analysis["file_name"] = malicious_analysis["message"].str.extract(
    r"File:\s*(.*?)\s*\|\s*Quarantined At:"
)[0]

malicious_analysis["quarantined_at"] = malicious_analysis["message"].str.extract(
    r"Quarantined At:\s*([^|]+)"
)[0]

malicious_analysis["sha256"] = malicious_analysis["message"].str.extract(
    r"SHA256:\s*([A-Fa-f0-9]{64})"
)[0]

malicious_analysis["quarantined_at"] = pd.to_datetime(
    malicious_analysis["quarantined_at"],
    errors="coerce"
)

print("Records:", len(malicious_analysis))
print("Unique files:", malicious_analysis["file_name"].nunique())
print("Unique SHA256:", malicious_analysis["sha256"].nunique())
print("Missing SHA256:", malicious_analysis["sha256"].isna().sum())

Records: 134
Unique files: 54
Unique SHA256: 51
Missing SHA256: 0


In [9]:
artifact_summary = (
    malicious_analysis
    .groupby(["endpoint_id", "file_name", "sha256"], dropna=False)
    .agg(
        detections=("endpoint_id", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max")
    )
    .reset_index()
    .sort_values(
        ["detections", "endpoint_id"],
        ascending=[False, True]
    )
)

display(artifact_summary)

,endpoint_id,file_name,sha256,detections,first_event,last_event
26,EPC2L3UH,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,18,2026-02-09 08:48:22,2026-12-09 09:15:26
35,EPJYEC7F,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,10,2026-01-09 08:09:22,2026-07-09 09:03:50
44,EPOC6Y4R,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,2026-01-09 08:59:34,2026-04-09 08:23:22
45,EPPGWIOF,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
46,EPPGWIOF,rawtobr3,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
...,...,...,...,...,...,...
59,EPVSWBNA,Strings.dll,ae107170c153cb3eb34c049482722d3937f2c03a6c23ca...,1,NaT,NaT
60,EPVSWBNA,launcher.exe,61a67e53c6c0c80a4e95af35a2997ad2ce59720020fc74...,1,NaT,NaT
61,EPVSWBNA,zshp1020s.dll,c6525c5230387653e16f21f408eab8202157b0eddeb180...,1,NaT,NaT
62,EPWZ7L6L,ops-electron-app.desktop,dd294df9886197bd68807cee3bf62d10859b4860bd9542...,1,NaT,NaT


In [10]:
artifact_summary = (
    malicious_analysis
    .groupby(["endpoint_id", "file_name", "sha256"], dropna=False)
    .agg(
        detections=("endpoint_id", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max")
    )
    .reset_index()
    .sort_values(
        ["detections", "endpoint_id"],
        ascending=[False, True]
    )
)

print("Unique endpoint/file/hash combinations:", len(artifact_summary))

display(artifact_summary.head(30))

Unique endpoint/file/hash combinations: 64


,endpoint_id,file_name,sha256,detections,first_event,last_event
26,EPC2L3UH,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,18,2026-02-09 08:48:22,2026-12-09 09:15:26
35,EPJYEC7F,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,10,2026-01-09 08:09:22,2026-07-09 09:03:50
44,EPOC6Y4R,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,2026-01-09 08:59:34,2026-04-09 08:23:22
45,EPPGWIOF,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
46,EPPGWIOF,rawtobr3,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
34,EPJ76RZV,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,4,NaT,NaT
1,EP3GALVS,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,2,NaT,NaT
2,EP4YIFHV,app-setup.js,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
3,EP4YIFHV,bnpl.bundle.js,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
4,EP4YIFHV,bnpl_driver.js,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [11]:
artifact_summary.head(30)

,endpoint_id,file_name,sha256,detections,first_event,last_event
26,EPC2L3UH,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,18,2026-02-09 08:48:22,2026-12-09 09:15:26
35,EPJYEC7F,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,10,2026-01-09 08:09:22,2026-07-09 09:03:50
44,EPOC6Y4R,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,2026-01-09 08:59:34,2026-04-09 08:23:22
45,EPPGWIOF,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
46,EPPGWIOF,rawtobr3,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
34,EPJ76RZV,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,4,NaT,NaT
1,EP3GALVS,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,2,NaT,NaT
2,EP4YIFHV,app-setup.js,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
3,EP4YIFHV,bnpl.bundle.js,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
4,EP4YIFHV,bnpl_driver.js,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [12]:
endpoint_artifacts = (
    malicious_analysis
    .groupby("endpoint_id")
    .agg(
        malicious_records=("endpoint_id", "size"),
        unique_files=("file_name", "nunique"),
        unique_hashes=("sha256", "nunique")
    )
    .reset_index()
    .sort_values(
        ["unique_hashes", "malicious_records"],
        ascending=False
    )
)

print("Endpoints:", len(endpoint_artifacts))

display(endpoint_artifacts)

Endpoints: 33


,endpoint_id,malicious_records,unique_files,unique_hashes
2,EP4YIFHV,40,20,20
18,EPMVZYZO,4,4,4
30,EPVSWBNA,3,3,3
21,EPPGWIOF,10,2,2
5,EPAJI3E5,4,2,2
12,EPJ42VCZ,2,2,2
6,EPC2L3UH,18,1,1
14,EPJYEC7F,10,1,1
20,EPOC6Y4R,9,1,1
26,EPSES7ZK,5,5,1


In [13]:
hash_distribution = (
    malicious_analysis
    .groupby("sha256")
    .agg(
        endpoint_count=("endpoint_id", "nunique"),
        endpoints=("endpoint_id", lambda x: ", ".join(sorted(x.unique()))),
        file_count=("file_name", "nunique"),
        files=("file_name", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values(
        ["endpoint_count", "file_count"],
        ascending=False
    )
)

display(
    hash_distribution[
        hash_distribution["endpoint_count"] > 1
    ]
)

,sha256,endpoint_count,endpoints,file_count,files
16,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,"EP3GALVS, EPC2L3UH, EPDRB236, EPFQSCDV, EPJYEC...",1,recently-used.xbel
30,93ffd792f92437a850e4939c84442391a624e20446cea2...,2,"EP2SJ6NE, EPF4AJCM",1,libreoffice-calc.desktop


In [14]:
print("event_time dtype:", malicious["event_time"].dtype)

display(
    malicious[
        ["endpoint_id", "event_time"]
    ]
    .sort_values("event_time")
    .head(20)
)

print("\nSample EPC2L3UH:")
display(
    malicious[
        malicious["endpoint_id"] == "EPC2L3UH"
    ][["endpoint_id", "event_time", "message"]]
    .sort_values("event_time")
)

event_time dtype: datetime64[us]


,endpoint_id,event_time
1,EPZRAPEL,2026-01-09 04:58:58
995,EPJYEC7F,2026-01-09 08:09:22
394,EPOC6Y4R,2026-01-09 08:59:34
671,EPOC6Y4R,2026-01-09 18:21:26
1045,EPMQTKSL,2026-02-09 08:37:11
1050,EPOC6Y4R,2026-02-09 08:39:53
1148,EPC2L3UH,2026-02-09 08:48:22
3415,EPAGTKNF,2026-03-09 19:03:33
3471,EPOC6Y4R,2026-03-09 19:14:05
2843,EPOC6Y4R,2026-04-09 08:23:22



Sample EPC2L3UH:


,endpoint_id,event_time,message
1148,EPC2L3UH,2026-02-09 08:48:22,File: recently-used.xbel | Quarantined At: 202...
3801,EPC2L3UH,2026-05-09 08:58:56,File: recently-used.xbel | Quarantined At: 202...
6215,EPC2L3UH,2026-08-09 08:41:56,File: recently-used.xbel | Quarantined At: 202...
6914,EPC2L3UH,2026-09-09 08:40:46,File: recently-used.xbel | Quarantined At: 202...
7625,EPC2L3UH,2026-09-09 17:22:33,File: recently-used.xbel | Quarantined At: 202...
8307,EPC2L3UH,2026-10-09 08:42:46,File: recently-used.xbel | Quarantined At: 202...
9936,EPC2L3UH,2026-12-09 09:15:26,File: recently-used.xbel | Quarantined At: 202...
11892,EPC2L3UH,NaT,File: recently-used.xbel | Quarantined At: 202...
12413,EPC2L3UH,NaT,File: recently-used.xbel | Quarantined At: 202...
12934,EPC2L3UH,NaT,File: recently-used.xbel | Quarantined At: 202...


In [15]:
print("Min event:", malicious["event_time"].min())
print("Max event:", malicious["event_time"].max())

print("\nDtype:")
print(malicious["event_time"].dtype)

Min event: 2026-01-09 04:58:58
Max event: 2026-12-09 09:15:26

Dtype:
datetime64[us]


In [18]:
# Start from the raw malicious quarantine records
malicious_clean = malicious.copy()

# --------------------------------------------------
# Extract artifact information from the message
# --------------------------------------------------

# File name
malicious_clean["file_name"] = malicious_clean["message"].str.extract(
    r"File:\s*(.*?)\s*\|\s*Quarantined At:",
    expand=False
)

# SHA256
malicious_clean["sha256"] = malicious_clean["message"].str.extract(
    r"SHA256:\s*([A-Fa-f0-9]{64})",
    expand=False
)

# --------------------------------------------------
# Validate
# --------------------------------------------------

print("Columns:")
print(malicious_clean.columns.tolist())

print("\nRecords:", len(malicious_clean))
print("Endpoints:", malicious_clean["endpoint_id"].nunique())
print("Unique files:", malicious_clean["file_name"].nunique())
print("Unique SHA256:", malicious_clean["sha256"].nunique())

print("\nMissing file_name:", malicious_clean["file_name"].isna().sum())
print("Missing SHA256:", malicious_clean["sha256"].isna().sum())
print("Missing event_time:", malicious_clean["event_time"].isna().sum())

display(
    malicious_clean[
        ["endpoint_id", "event_time", "file_name", "sha256"]
    ].head(20)
)

Columns:
['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message', 'file_name', 'sha256']

Records: 134
Endpoints: 33
Unique files: 54
Unique SHA256: 51

Missing file_name: 0
Missing SHA256: 0
Missing event_time: 55


,endpoint_id,event_time,file_name,sha256
1,EPZRAPEL,2026-01-09 04:58:58,53EDA42869D426262B752F088EA605187284ACE8,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...
394,EPOC6Y4R,2026-01-09 08:59:34,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
671,EPOC6Y4R,2026-01-09 18:21:26,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
995,EPJYEC7F,2026-01-09 08:09:22,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
1045,EPMQTKSL,2026-02-09 08:37:11,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
1050,EPOC6Y4R,2026-02-09 08:39:53,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
1148,EPC2L3UH,2026-02-09 08:48:22,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
2843,EPOC6Y4R,2026-04-09 08:23:22,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
2997,EPAGTKNF,2026-04-09 18:10:01,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ...",18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...
3415,EPAGTKNF,2026-03-09 19:03:33,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ...",18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...


In [19]:
# Endpoint-level artifact summary
artifact_summary = (
    malicious_clean
    .groupby("endpoint_id")
    .agg(
        malicious_records=("endpoint_id", "size"),
        unique_files=("file_name", "nunique"),
        unique_hashes=("sha256", "nunique")
    )
    .sort_values(
        ["unique_hashes", "malicious_records"],
        ascending=False
    )
)

display(artifact_summary)

,malicious_records,unique_files,unique_hashes
endpoint_id,,,
EP4YIFHV,40,20,20
EPMVZYZO,4,4,4
EPVSWBNA,3,3,3
EPPGWIOF,10,2,2
EPAJI3E5,4,2,2
EPJ42VCZ,2,2,2
EPC2L3UH,18,1,1
EPJYEC7F,10,1,1
EPOC6Y4R,9,1,1


In [20]:
# Cross-endpoint hash reuse
hash_reuse = (
    malicious_clean
    .groupby("sha256")
    .agg(
        endpoint_count=("endpoint_id", "nunique"),
        endpoints=("endpoint_id", lambda x: ", ".join(sorted(x.unique()))),
        file_count=("file_name", "nunique"),
        files=("file_name", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

display(
    hash_reuse[
        hash_reuse["endpoint_count"] > 1
    ]
    .sort_values("endpoint_count", ascending=False)
)

,sha256,endpoint_count,endpoints,file_count,files
16,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,"EP3GALVS, EPC2L3UH, EPDRB236, EPFQSCDV, EPJYEC...",1,recently-used.xbel
30,93ffd792f92437a850e4939c84442391a624e20446cea2...,2,"EP2SJ6NE, EPF4AJCM",1,libreoffice-calc.desktop


In [21]:
endpoint_files = (
    malicious_clean
    .groupby("endpoint_id")
    .agg(
        records=("endpoint_id", "size"),
        files=("file_name", lambda x: sorted(x.unique().tolist())),
        hashes=("sha256", lambda x: sorted(x.unique().tolist()))
    )
    .reset_index()
)

endpoint_files["file_count"] = endpoint_files["files"].str.len()
endpoint_files["hash_count"] = endpoint_files["hashes"].str.len()

display(
    endpoint_files[
        ["endpoint_id", "records", "file_count", "hash_count", "files"]
    ].sort_values(
        ["file_count", "records"],
        ascending=False
    )
)

,endpoint_id,records,file_count,hash_count,files
2,EP4YIFHV,40,20,20,"[app-setup.js, bnpl.bundle.js, bnpl_driver.js,..."
26,EPSES7ZK,5,5,1,[hp_LJ1020_Full_Solution-v2012_918_1_57980(1)....
18,EPMVZYZO,4,4,4,"[mydoc-10.pdf, mydoc-75.pdf, mydoc-9.pdf, mydo..."
30,EPVSWBNA,3,3,3,"[Strings.dll, launcher.exe, zshp1020s.dll]"
21,EPPGWIOF,10,2,2,"[brprintconflsr3, rawtobr3]"
5,EPAJI3E5,4,2,2,"[Lakeside traders.docx, Lakeside traders.pdf]"
12,EPJ42VCZ,2,2,2,"[RAGHAV STNA STNB.pdf, RAGHAV STNC STNA.pdf]"
6,EPC2L3UH,18,1,1,[recently-used.xbel]
14,EPJYEC7F,10,1,1,[recently-used.xbel]
20,EPOC6Y4R,9,1,1,[recently-used.xbel]


In [22]:
# Show every artifact that occurs on more than one endpoint
artifact_cross_endpoint = (
    malicious_clean
    .groupby(["file_name", "sha256"])
    .agg(
        endpoint_count=("endpoint_id", "nunique"),
        endpoints=("endpoint_id", lambda x: sorted(x.unique().tolist())),
        record_count=("endpoint_id", "size")
    )
    .reset_index()
)

display(
    artifact_cross_endpoint[
        artifact_cross_endpoint["endpoint_count"] > 1
    ].sort_values(
        ["endpoint_count", "record_count"],
        ascending=False
    )
)

,file_name,sha256,endpoint_count,endpoints,record_count
45,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,9,"[EP3GALVS, EPC2L3UH, EPDRB236, EPFQSCDV, EPJYE...",45
31,libreoffice-calc.desktop,93ffd792f92437a850e4939c84442391a624e20446cea2...,2,"[EP2SJ6NE, EPF4AJCM]",2


In [23]:
# Remove only the two clearly cross-endpoint artifacts
cross_endpoint_hashes = artifact_cross_endpoint.loc[
    artifact_cross_endpoint["endpoint_count"] > 1,
    "sha256"
]

after_cross_endpoint = malicious_clean[
    ~malicious_clean["sha256"].isin(cross_endpoint_hashes)
].copy()

print("Records after cross-endpoint artifact removal:",
      len(after_cross_endpoint))

print("Endpoints remaining:",
      after_cross_endpoint["endpoint_id"].nunique())

print("\nRemaining endpoint summary:")
display(
    after_cross_endpoint
    .groupby("endpoint_id")
    .agg(
        records=("endpoint_id", "size"),
        files=("file_name", "nunique"),
        hashes=("sha256", "nunique")
    )
    .sort_values("records", ascending=False)
)

Records after cross-endpoint artifact removal: 87
Endpoints remaining: 22

Remaining endpoint summary:


,records,files,hashes
endpoint_id,,,
EP4YIFHV,40,20,20
EPPGWIOF,10,2,2
EPSES7ZK,5,5,1
EPMVZYZO,4,4,4
EPJ76RZV,4,1,1
EPAJI3E5,4,2,2
EPVSWBNA,3,3,3
EPJ42VCZ,2,2,2
EPAGTKNF,2,1,1


In [24]:
# Inspect all remaining artifacts, especially the small/one-off cases

remaining_artifacts = (
    after_cross_endpoint
    .groupby(["endpoint_id", "file_name", "sha256"])
    .agg(
        records=("endpoint_id", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max")
    )
    .reset_index()
    .sort_values(
        ["records", "endpoint_id"],
        ascending=[False, True]
    )
)

display(remaining_artifacts)

,endpoint_id,file_name,sha256,records,first_event,last_event
35,EPPGWIOF,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
36,EPPGWIOF,rawtobr3,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,5,2026-07-09 11:25:38,2026-08-09 08:12:56
28,EPJ76RZV,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,4,NaT,NaT
0,EP4YIFHV,app-setup.js,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
1,EP4YIFHV,bnpl.bundle.js,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
2,EP4YIFHV,bnpl_driver.js,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
3,EP4YIFHV,buynow_driver.js,bee346d14d293f9f4b989362041875b0257477a7775985...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
4,EP4YIFHV,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
5,EP4YIFHV,crypto.bundle.js,8e661de94430dc0293ca7964f0f9b89682e8fd2f56c17f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
6,EP4YIFHV,edge_driver.js,985d0862ba672e36839438e44cbaf6d18537b53010b778...,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [25]:
artifact_level = (
    after_cross_endpoint
    .groupby(
        ["endpoint_id", "file_name", "sha256"],
        dropna=False
    )
    .agg(
        record_count=("endpoint_id", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max")
    )
    .reset_index()
)

# Number of endpoints associated with each hash
hash_endpoint_counts = (
    after_cross_endpoint
    .groupby("sha256")["endpoint_id"]
    .nunique()
    .rename("endpoint_count")
)

artifact_level = artifact_level.merge(
    hash_endpoint_counts,
    on="sha256",
    how="left"
)

# Classify artifact repetition
artifact_level["cross_endpoint"] = (
    artifact_level["endpoint_count"] > 1
)

artifact_level["repeated_on_endpoint"] = (
    artifact_level["record_count"] > 1
)

artifact_level = artifact_level.sort_values(
    ["record_count", "endpoint_count"],
    ascending=[False, False]
)

print("Artifact-level rows:", len(artifact_level))

display(artifact_level)

Artifact-level rows: 53


,endpoint_id,file_name,sha256,record_count,first_event,last_event,endpoint_count,cross_endpoint,repeated_on_endpoint
35,EPPGWIOF,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,5,2026-07-09 11:25:38,2026-08-09 08:12:56,1,False,True
36,EPPGWIOF,rawtobr3,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,5,2026-07-09 11:25:38,2026-08-09 08:12:56,1,False,True
28,EPJ76RZV,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,4,NaT,NaT,1,False,True
0,EP4YIFHV,app-setup.js,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
1,EP4YIFHV,bnpl.bundle.js,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
2,EP4YIFHV,bnpl_driver.js,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
3,EP4YIFHV,buynow_driver.js,bee346d14d293f9f4b989362041875b0257477a7775985...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
4,EP4YIFHV,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
5,EP4YIFHV,crypto.bundle.js,8e661de94430dc0293ca7964f0f9b89682e8fd2f56c17f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True
6,EP4YIFHV,edge_driver.js,985d0862ba672e36839438e44cbaf6d18537b53010b778...,2,2026-11-09 09:23:33,2026-11-09 09:35:09,1,False,True


In [26]:
print("Total records:", len(after_cross_endpoint))
print("Artifact-level rows:", len(artifact_level))

print(
    "\nArtifacts repeated on same endpoint:",
    artifact_level["repeated_on_endpoint"].sum()
)

print(
    "Artifacts shared across endpoints:",
    artifact_level["cross_endpoint"].sum()
)

print(
    "Artifacts occurring exactly once:",
    (~artifact_level["repeated_on_endpoint"]).sum()
)

Total records: 87
Artifact-level rows: 53

Artifacts repeated on same endpoint: 26
Artifacts shared across endpoints: 0
Artifacts occurring exactly once: 27


In [29]:
artifact_summary = (
    artifact_level
    .groupby("endpoint_id")
    .agg(
        artifact_count=("file_name", "size"),
        unique_hashes=("sha256", "nunique"),
        detection_records=("record_count", "sum"),
        repeated_artifacts=("repeated_on_endpoint", "sum")
    )
    .reset_index()
    .sort_values(
        ["artifact_count", "detection_records"],
        ascending=False
    )
)

display(artifact_summary)

,endpoint_id,artifact_count,unique_hashes,detection_records,repeated_artifacts
0,EP4YIFHV,20,20,40,20
15,EPSES7ZK,5,1,5,0
9,EPMVZYZO,4,4,4,0
19,EPVSWBNA,3,3,3,0
11,EPPGWIOF,2,2,10,2
3,EPAJI3E5,2,2,4,2
6,EPJ42VCZ,2,2,2,0
7,EPJ76RZV,1,1,4,1
2,EPAGTKNF,1,1,2,1
1,EP7UYCXM,1,1,1,0


In [30]:
print("Total endpoints:", artifact_summary["endpoint_id"].nunique())
print("Total artifacts:", artifact_summary["artifact_count"].sum())
print("Total records:", artifact_summary["detection_records"].sum())

Total endpoints: 22
Total artifacts: 53
Total records: 87


In [31]:
artifact_level["extension"] = (
    artifact_level["file_name"]
    .astype(str)
    .str.extract(r"(\.[A-Za-z0-9]+)$", expand=False)
    .str.lower()
    .fillna("[no extension]")
)

extension_summary = (
    artifact_level
    .groupby("extension")
    .agg(
        artifacts=("sha256", "nunique"),
        records=("record_count", "sum"),
        endpoints=("endpoint_id", "nunique")
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

display(extension_summary)

,extension,artifacts,records,endpoints
6,.js,19,38,1
7,.pdf,11,13,7
11,[no extension],4,13,3
5,.exe,3,7,3
9,.xlsx,2,5,2
4,.docx,2,3,2
3,.dll,3,3,2
0,.77,1,1,1
2,.desktop,1,1,1
1,.cab,1,1,1


In [32]:
endpoint_artifacts = (
    artifact_level[
        [
            "endpoint_id",
            "file_name",
            "sha256",
            "record_count",
            "first_event",
            "last_event"
        ]
    ]
    .sort_values(
        ["endpoint_id", "record_count"],
        ascending=[True, False]
    )
)

display(endpoint_artifacts)

,endpoint_id,file_name,sha256,record_count,first_event,last_event
0,EP4YIFHV,app-setup.js,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
1,EP4YIFHV,bnpl.bundle.js,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
2,EP4YIFHV,bnpl_driver.js,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
3,EP4YIFHV,buynow_driver.js,bee346d14d293f9f4b989362041875b0257477a7775985...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
4,EP4YIFHV,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
5,EP4YIFHV,crypto.bundle.js,8e661de94430dc0293ca7964f0f9b89682e8fd2f56c17f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
6,EP4YIFHV,edge_driver.js,985d0862ba672e36839438e44cbaf6d18537b53010b778...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
7,EP4YIFHV,load-ec-deps.bundle.js,f2bf148b4d999cb41ec764960ee718e0c9986a749f3b7b...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
8,EP4YIFHV,load-ec-i18n.bundle.js,56bada50510d97878d514009279c3a662d9894a21c344f...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
9,EP4YIFHV,load-hub-i18n.bundle.js,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [33]:
print("Rows:", len(endpoint_artifacts))
print("Endpoints:", endpoint_artifacts["endpoint_id"].nunique())
print("Records:", endpoint_artifacts["record_count"].sum())

Rows: 53
Endpoints: 22
Records: 87


In [34]:
# Graph-ready artifact table
artifact_nodes = (
    artifact_level[
        [
            "sha256",
            "file_name",
            "record_count",
            "first_event",
            "last_event"
        ]
    ]
    .groupby("sha256", as_index=False)
    .agg(
        file_name=("file_name", "first"),
        record_count=("record_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max")
    )
)

print("Artifact nodes:", len(artifact_nodes))
print("Unique SHA256:", artifact_nodes["sha256"].nunique())

display(artifact_nodes)

Artifact nodes: 49
Unique SHA256: 49


,sha256,file_name,record_count,first_event,last_event
0,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,load-hub-i18n.bundle.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
1,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,shopping_iframe_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
2,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ...",2,2026-03-09 19:03:33,2026-04-09 18:10:01
3,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,53EDA42869D426262B752F088EA605187284ACE8,1,2026-01-09 04:58:58,2026-01-09 04:58:58
4,30c78bc9ec28a6e4d47708c309f8c88599c71a0fb379a3...,dash cam.pdf,1,NaT,NaT
5,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,tokenized-card.bundle.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
6,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...,Final_Documents.zip,1,2026-11-09 10:11:45,2026-11-09 10:11:45
7,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,wallet_checkout_autofill_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
8,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,wallet-buynow.bundle.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
9,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,wallet_donation_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [35]:
endpoint_artifact_edges = (
    artifact_level[
        [
            "endpoint_id",
            "sha256",
            "file_name",
            "record_count",
            "first_event",
            "last_event"
        ]
    ]
    .copy()
)

print("Endpoint-artifact edges:", len(endpoint_artifact_edges))

display(endpoint_artifact_edges)

Endpoint-artifact edges: 53


,endpoint_id,sha256,file_name,record_count,first_event,last_event
35,EPPGWIOF,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,brprintconflsr3,5,2026-07-09 11:25:38,2026-08-09 08:12:56
36,EPPGWIOF,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,rawtobr3,5,2026-07-09 11:25:38,2026-08-09 08:12:56
28,EPJ76RZV,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,4,NaT,NaT
0,EP4YIFHV,d604ac767e070cf98b235b0281ef932c463fa11bceaf2f...,app-setup.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
1,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,bnpl.bundle.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
2,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,bnpl_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
3,EP4YIFHV,bee346d14d293f9f4b989362041875b0257477a7775985...,buynow_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
4,EP4YIFHV,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,2,2026-11-09 09:23:33,2026-11-09 09:35:09
5,EP4YIFHV,8e661de94430dc0293ca7964f0f9b89682e8fd2f56c17f...,crypto.bundle.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09
6,EP4YIFHV,985d0862ba672e36839438e44cbaf6d18537b53010b778...,edge_driver.js,2,2026-11-09 09:23:33,2026-11-09 09:35:09


In [36]:
artifact_nodes = (
    artifact_level
    .groupby("sha256", as_index=False)
    .agg(
        file_names=("file_name", lambda x: sorted(set(x.dropna()))),
        record_count=("record_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max")
    )
)

artifact_nodes["file_name_count"] = artifact_nodes["file_names"].apply(len)

print("Artifact nodes:", len(artifact_nodes))
print("Unique SHA256:", artifact_nodes["sha256"].nunique())

display(artifact_nodes)

Artifact nodes: 49
Unique SHA256: 49


,sha256,file_names,record_count,first_event,last_event,file_name_count
0,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
1,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
2,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"[Buy CP PLUS Dashboard Camera, Warranty NONE 1...",2,2026-03-09 19:03:33,2026-04-09 18:10:01,1
3,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,[53EDA42869D426262B752F088EA605187284ACE8],1,2026-01-09 04:58:58,2026-01-09 04:58:58,1
4,30c78bc9ec28a6e4d47708c309f8c88599c71a0fb379a3...,[dash cam.pdf],1,NaT,NaT,1
5,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
6,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...,[Final_Documents.zip],1,2026-11-09 10:11:45,2026-11-09 10:11:45,1
7,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
8,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
9,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1


In [37]:
endpoint_artifact_edges = (
    artifact_level
    .groupby(
        ["endpoint_id", "sha256"],
        as_index=False
    )
    .agg(
        file_names=("file_name", lambda x: sorted(set(x.dropna()))),
        record_count=("record_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max")
    )
)

endpoint_artifact_edges["file_name_count"] = (
    endpoint_artifact_edges["file_names"].apply(len)
)

print("Unique endpoint-artifact relationships:",
      len(endpoint_artifact_edges))

display(endpoint_artifact_edges)

Unique endpoint-artifact relationships: 49


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1


In [38]:
artifact_nodes = (
    artifact_level
    .groupby("sha256", as_index=False)
    .agg(
        file_names=("file_name", lambda x: sorted(set(x.dropna()))),
        record_count=("record_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max")
    )
)

artifact_nodes["file_name_count"] = (
    artifact_nodes["file_names"].apply(len)
)

print("Artifacts:", len(artifact_nodes))
print("Total records:", artifact_nodes["record_count"].sum())

display(artifact_nodes)

Artifacts: 49
Total records: 87


,sha256,file_names,record_count,first_event,last_event,file_name_count
0,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
1,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
2,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"[Buy CP PLUS Dashboard Camera, Warranty NONE 1...",2,2026-03-09 19:03:33,2026-04-09 18:10:01,1
3,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,[53EDA42869D426262B752F088EA605187284ACE8],1,2026-01-09 04:58:58,2026-01-09 04:58:58,1
4,30c78bc9ec28a6e4d47708c309f8c88599c71a0fb379a3...,[dash cam.pdf],1,NaT,NaT,1
5,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
6,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...,[Final_Documents.zip],1,2026-11-09 10:11:45,2026-11-09 10:11:45,1
7,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
8,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
9,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1


In [39]:
endpoint_nodes = (
    artifact_level[["endpoint_id"]]
    .drop_duplicates()
    .sort_values("endpoint_id")
    .reset_index(drop=True)
)

print("Endpoints:", len(endpoint_nodes))

display(endpoint_nodes)

Endpoints: 22


,endpoint_id
0,EP4YIFHV
1,EP7UYCXM
2,EPAGTKNF
3,EPAJI3E5
4,EPEEHWSG
5,EPGSQVEB
6,EPJ42VCZ
7,EPJ76RZV
8,EPMDU6L5
9,EPMVZYZO


In [40]:
endpoint_artifact_edges = (
    artifact_level
    .groupby(
        ["endpoint_id", "sha256"],
        as_index=False
    )
    .agg(
        file_names=("file_name", lambda x: sorted(set(x.dropna()))),
        record_count=("record_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max")
    )
)

endpoint_artifact_edges["file_name_count"] = (
    endpoint_artifact_edges["file_names"].apply(len)
)

print(
    "Unique Endpoint → Artifact edges:",
    len(endpoint_artifact_edges)
)

display(endpoint_artifact_edges)

Unique Endpoint → Artifact edges: 49


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-11-09 09:23:33,2026-11-09 09:35:09,1


In [41]:
def get_extension(filename):
    if not filename or pd.isna(filename):
        return "[no extension]"

    filename = str(filename).strip()

    if "." not in filename:
        return "[no extension]"

    return "." + filename.rsplit(".", 1)[1].lower()


artifact_level["extension"] = (
    artifact_level["file_name"]
    .apply(get_extension)
)

display(
    artifact_level[
        ["file_name", "extension"]
    ].drop_duplicates()
)

,file_name,extension
35,brprintconflsr3,[no extension]
36,rawtobr3,[no extension]
28,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,.xlsx
0,app-setup.js,.js
1,bnpl.bundle.js,.js
2,bnpl_driver.js,.js
3,buynow_driver.js,.js
4,c3f9c4a416e9e493005fc3f97adacbcf12956a81cc6c64...,[no extension]
5,crypto.bundle.js,.js
6,edge_driver.js,.js


In [42]:
print("========== GRAPH VALIDATION ==========")

print(
    "Endpoints:",
    endpoint_nodes["endpoint_id"].nunique()
)

print(
    "Artifacts:",
    artifact_nodes["sha256"].nunique()
)

print(
    "Endpoint-Artifact edges:",
    len(endpoint_artifact_edges)
)

print(
    "Artifact records:",
    artifact_nodes["record_count"].sum()
)

print(
    "Original records:",
    87
)

print(
    "Original artifact-level rows:",
    len(artifact_level)
)

========== GRAPH VALIDATION ==========
Endpoints: 22
Artifacts: 49
Endpoint-Artifact edges: 49
Artifact records: 87
Original records: 87
Original artifact-level rows: 53


In [43]:
assert endpoint_nodes["endpoint_id"].nunique() == 22
assert artifact_nodes["sha256"].nunique() == 49
assert len(endpoint_artifact_edges) == 49
assert artifact_nodes["record_count"].sum() == 87

In [44]:
print(
    artifact_nodes["record_count"]
    .value_counts()
    .sort_index()
)

record_count
1    22
2    23
4     1
5     3
Name: count, dtype: int64


In [45]:
print(
    "Repeated artifacts:",
    (artifact_nodes["record_count"] > 1).sum()
)

print(
    "Single-occurrence artifacts:",
    (artifact_nodes["record_count"] == 1).sum()
)

Repeated artifacts: 27
Single-occurrence artifacts: 22
